In [ ]:
import numpy as np
from pathlib import Path

import prism

from imagematerials.factory import ModelFactory, Sector
from imagematerials.model import (
    GenericStocks,
    StocksQuintiles,
    SharesInflowStocks,
    ElectricVehicleBatteries,
    GenericMaterials,
    MaterialIntensities,
    MaterialIntensitiesQuintiles,
    RestOf,
    EndOfLife
)
from imagematerials.preprocessing import get_preprocessing_data
from imagematerials.util import resolve_circular_economy_scenario
from imagematerials.concepts import create_electricity_graph
from imagematerials.electricity.constants import EPG_TECHNOLOGIES_FINAL
from imagematerials.electricity.utils import rebroadcast_type_dims
knowledge_graph_electricity = create_electricity_graph()

import warnings
from pint import UnitStrippedWarning

warnings.filterwarnings("ignore", category=UnitStrippedWarning)

base_path = Path("..", "data", "raw")

scenario_list = {
    "SSP2_baseline_base":    ("SSP2_baseline", "base"),
    "SSP2_baseline_full_ce": ("SSP2_baseline", "full_ce"),
                 }

climate_scenarios = list(scenario_list.keys())

# run model

In [ ]:
# Define the complete timeline, including historic tail
time_start = 1971
time_end = 2100
complete_timeline = prism.Timeline(time_start, time_end, 1)
simulation_timeline = prism.Timeline(time_start, time_end, 1)

model_runs = {}

for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    print(f"Running scenario: {scen_id}")
    print(f"Circular economy scenario: {circular_scen}")

    climate_policy_scenario_dir = base_path / Path("image", climate_scen)
    circular_economy_scenario_dir = resolve_circular_economy_scenario(
        base_path / "circular_economy_scenarios", circular_scen)

    # BUILDINGS
    bld_sector = get_preprocessing_data("buildings",
                                        base_path,
                                        climate_policy_scenario_dir,
                                        circular_economy_scenario_dir,
                                    resource_efficiency_flags_file=circular_economy_scenario_dir)
    # VEHICLES
    vhc_sector = get_preprocessing_data("vehicles",
                                        base_path,
                                        climate_policy_scenario_dir,
                                        circular_economy_scenario_dir,
                                    resource_efficiency_flags_file=circular_economy_scenario_dir)
    prep_data_vhc = vhc_sector.prep_data
    vhc_sector = Sector('vehicles', prep_data_vhc)

    ev_battery_sector = get_preprocessing_data("ev_battery",
                                               base_path,
                                               climate_policy_scenario_dir,
                                               circular_economy_scenario_dir,
                                    resource_efficiency_flags_file=circular_economy_scenario_dir)
    # ELECTRICITY
    elc_sector = get_preprocessing_data("electricity",
                                        base_path,
                                        climate_policy_scenario_dir,
                                        circular_economy_scenario_dir,
                                        cache = False,
                                    resource_efficiency_flags_file=circular_economy_scenario_dir)
    elc_gen, elc_grid_lines, elc_grid_add, elc_stor_phs, elc_stor_other = (
    {sec.name: sec for sec in elc_sector}[k]
    for k in ["elc_gen", "elc_grid_lines", "elc_grid_add", "elc_stor_phs", "elc_stor_other"])

    # REST OF SECTOR
    rest_sector = get_preprocessing_data("rest_of",
                                         base_path,
                                         climate_policy_scenario_dir,
                                         circular_economy_scenario_dir,
                                         scenario_name=climate_scen,
                                    resource_efficiency_flags_file=circular_economy_scenario_dir)

    factory = ModelFactory(
    [bld_sector, vhc_sector, ev_battery_sector, elc_gen, rest_sector,  # 
     elc_grid_lines, elc_grid_add, elc_stor_phs, elc_stor_other], complete_timeline
    ).add(StocksQuintiles, ["buildings"]
    ).add(GenericStocks, ["vehicles", "elc_grid_lines", "elc_grid_add", "elc_stor_phs"]
    ).add(ElectricVehicleBatteries, ["ev_battery"], input_sources={
        "stock_by_cohort": "vehicles",
        "inflow": "vehicles",
        "outflow_by_cohort": "vehicles"}
    ).add(SharesInflowStocks, ["elc_stor_other", "elc_gen"]
    ).add(MaterialIntensities, ["elc_gen", "elc_grid_lines", "elc_grid_add", 
                                "elc_stor_phs", "elc_stor_other"] 
    ).add(GenericMaterials,  "vehicles"
    ).add(MaterialIntensitiesQuintiles, "buildings",
    ).add(RestOf, "rest_of"
    )
    model = factory.finish()
    model_runs[scen_id] = model


    import warnings
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")
        model.simulate(simulation_timeline)

    # renaming Type coordinates (necessary due to work around within the sub-technology model for electricity generation)
    rebroadcast_type_dims(model.elc_gen, knowledge_graph_electricity, EPG_TECHNOLOGIES_FINAL)

## with EOL

In [ ]:
# Define the complete timeline, including historic tail
time_start = 1998
time_end = 2070
complete_timeline = prism.Timeline(time_start, time_end, 1)
simulation_timeline = prism.Timeline(time_start, time_end, 1)

model_runs = {}

for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    print(f"Running scenario: {scen_id}")
    print(f"Circular economy scenario: {circular_scen}")

    climate_policy_scenario_dir = base_path / Path("image", climate_scen)
    circular_economy_scenario_dir = resolve_circular_economy_scenario(
        base_path / "circular_economy_scenarios", circular_scen)


    # BUILDINGS
    bld_sector = get_preprocessing_data("buildings",
                                        base_path,
                                        climate_policy_scenario_dir,
                                        circular_economy_scenario_dir,
                                        resource_efficiency_flags_file=circular_economy_scenario_dir)
    # VEHICLES
    vhc_sector = get_preprocessing_data("vehicles",
                                        base_path,
                                        climate_policy_scenario_dir,
                                        circular_economy_scenario_dir,
                                        resource_efficiency_flags_file=circular_economy_scenario_dir)
    prep_data_vhc = vhc_sector.prep_data
    vhc_sector = Sector('vehicles', prep_data_vhc)

    ev_battery_sector = get_preprocessing_data("ev_battery",
                                               base_path,
                                               climate_policy_scenario_dir,
                                               circular_economy_scenario_dir,
                                                resource_efficiency_flags_file=circular_economy_scenario_dir)
    # ELECTRICITY
    elc_sector = get_preprocessing_data("electricity",
                                        base_path,
                                        climate_policy_scenario_dir,
                                        circular_economy_scenario_dir,
                                        cache = False,
                                        resource_efficiency_flags_file=circular_economy_scenario_dir)

    # FOSSIL FUELS
    ff_sector = get_preprocessing_data("fossil_fuels",
                                       base_path,
                                        climate_policy_scenario_dir=climate_policy_scenario_dir,
                                        circular_economy_data_file=circular_economy_scenario_dir,
                                        resource_efficiency_flags_file=circular_economy_scenario_dir) 
    # note: if ff transport should be included add "ff_trp" below + FlagIncludeTransportation in resource_efficiency_flags must be set to "true"

    # HYDROGEN PRODUCTION
    h2_sector = get_preprocessing_data("hydrogen",
                                       base_path,
                                        climate_policy_scenario_dir, 
                                        circular_economy_scenario_dir, cache = False,
                                        resource_efficiency_flags_file=circular_economy_scenario_dir) 

    # END OF LIFE
    eol_sector = get_preprocessing_data("eol",
                                        base_dir=base_path,
                                        climate_policy_scenario_dir=climate_policy_scenario_dir,
                                        circular_economy_data_file=circular_economy_scenario_dir,
                                        resource_efficiency_flags_file=circular_economy_scenario_dir)
    

    factory = ModelFactory(
    [bld_sector, vhc_sector, ev_battery_sector, *elc_sector, *ff_sector, h2_sector, eol_sector],
    complete_timeline
    ).add(StocksQuintiles, ["buildings"]
    ).add(GenericStocks, ["vehicles", "elc_grid_lines", "elc_grid_add", "elc_stor_phs",
                          "ff_extr","ff_proc","ff_pipe", "ff_str",
                          "hydrogen"]
    ).add(ElectricVehicleBatteries, ["ev_battery"], input_sources={
        "stock_by_cohort": "vehicles",
        "inflow": "vehicles",
        "outflow_by_cohort": "vehicles"}
    ).add(SharesInflowStocks, ["elc_stor_other", "elc_gen"]
    ).add(MaterialIntensities, ["elc_gen", "elc_grid_lines", "elc_grid_add", "elc_stor_phs", "elc_stor_other",
                                "ff_extr","ff_proc","ff_pipe", "ff_str",
                                "hydrogen"] 
    ).add(GenericMaterials,  "vehicles"
    ).add(MaterialIntensitiesQuintiles, "buildings",
    ).add(EndOfLife, "eol", input_sources={
        "outflow_by_cohort_materials": ["vehicles", "buildings",
                                        "elc_gen", "elc_grid_lines", "elc_grid_add", "elc_stor_phs",
                                        #"ev_battery",
                                        "hydrogen"
                                        ],
        "inflow_materials":  ["vehicles", "buildings",
                              "elc_gen", "elc_grid_lines", "elc_grid_add", "elc_stor_phs",
                              "hydrogen"
                             ],
        "collection": "eol",
        "reuse": "eol",
        "recycling": "eol"}
    )
    model = factory.finish()
    model_runs[scen_id] = model

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")
        model.simulate(simulation_timeline)

    # renaming Type coordinates (necessary due to work around within the sub-technology model for electricity generation)
    rebroadcast_type_dims(model.elc_gen, knowledge_graph_electricity, EPG_TECHNOLOGIES_FINAL)

In [ ]:
list(model.eol)

In [ ]:
eol_sector.prep_data["recycling"].sel(Type="fossil_fuel_infrastructure")

In [ ]:
model.eol["recyclable_materials"].to_array()

In [ ]:
model.eol["virgin_materials"].to_array()

In [ ]:
model.eol["recyclable_materials"].to_array()

# Plots

In [ ]:
import matplotlib.pyplot as plt

var1 = model.eol["virgin_materials"].to_array().sum("Region").sel(Type=["hydrogen"], material=["aluminium", "copper", "plastics"]) #, "passenger"
var2 = model.eol["sum_inflow"].to_array().sum("Region").sel(Type=["hydrogen"], material=["aluminium", "copper", "plastics"])

fig, ax = plt.subplots(figsize=(10, 6))

for type in var1.Type.values:
    for mat in var1.material.values:
        ax.plot(var1.time, var1.sel(Type=type, material=mat), label=f"vir. mat - {type} - {mat}")
        ax.plot(var2.time, var2.sel(Type=type, material=mat), label=f"inflow - {type} - {mat}")

ax.legend()
ax.grid(alpha=0.3)